> **For research and educational purposes only — not financial advice.** Backtest and analysis results are descriptive of historical data only and do not predict future market behaviour. See the README's Disclaimer section for full terms.

# 1. Core Pipeline — Fetch & Quantify

This notebook __produces the data__ used by the other two. It:
1. Fetches OHLC data from Bybit for a given symbol, interval, and date range.
2. Computes per-EMA metrics: touches, crosses, above/below counts, evaluated candles.
3. Caches OHLC to data/ as parquet (keyed by config). Notebooks 2 & 3 reuse it instead of re-fetching.
4. Writes data/config.json so notebooks 2 & 3 pick up the same parameters automatically — change them here, not there.

## Table of Contents

1. [Project Overview](#project-overview)
2. [Setup](#setup)
3. [Configuration](#configuration)
4. [Fetch OHLC](#fetch-ohlc)
5. [Cache and re-fetching](#cache-and-re-fetching)
6. [Aggregate EMA metrics](#aggregate-ema-metrics)
7. [Sanity check](#sanity-check)
8. [Data export](#data-export)
9. [Next steps](#next-steps)

<a id="project-overview"></a>
## Project Overview

__Data:__ \
Bybit Perpetual Futures.

__Configurable parameters:__
- symbol
- interval
- start/end dates
- ema_range
- delta
- delta_mode
- category

The configuration block in this notebook is the **single source of truth**: save_config stores the settings in data/config.json, and notebooks 2 & 3 call load_config to read it back.

__Workflow:__ \
For each EMA period in a user-supplied range, every evaluated candle is placed in exactly one of five mutually-exclusive categories, so the categories always sum to evaluated_candles.

__Categories__ (mutually exclusive partition; every candle lands in one category):
- cross      → Low ≤ EMA ≤ High. The candle's range straddles the EMA — the level was *broken*, not held.
- low_touch  → Low > EMA AND Low − EMA ≤ delta. Entire candle above EMA; low approached EMA from above without crossing — a *test of support*.
- high_touch → High < EMA AND EMA − High ≤ delta. Entire candle below EMA; high approached EMA from below without crossing — a *test of resistance*.
- above      → Low > EMA AND Low − EMA > delta. Entire candle above EMA, low far away — clean uptrend, no test.
- below      → High < EMA AND EMA − High > delta. Entire candle below EMA, high far away — clean downtrend, no test.

__Invariants:__
- cross + low_touch + high_touch + above + below = evaluated_candles
- any_touch = low_touch + high_touch (the two are mutually exclusive — a candle either tests support OR resistance, never both)

__Notes:__
- The first period candles are skipped per EMA (warm-up); toggle with skip_warmup=False if you don't want that.
- Pagination handles arbitrarily long date ranges (1000-candle chunks).
- Uses ewm(span=period, adjust=False) — standard Wilder/TradingView-style EMA.


<a id="setup"></a>
## Setup

In [1]:
from ema_core import load_or_fetch_klines, analyze_ema_touches, save_config

<a id="configuration"></a>
## Configuration

Adjust all parameters to fit your market.

save_config stores these settings in data/config.json, so notebooks 2 & 3 load them automatically — no edits needed there.

To perform analysis & backtesting with __new parameters__:
- edit values in this notebook
- re-run this notebook to update
- the other notebooks pick them up automatically on their next run

Parameters:
- symbol
  - examples: "BTCUSDT", "SOLUSDT", "ETHUSDT", "XRPUSDT", "DOGEUSDT", "BTCUSD-INVERSE", etc.
- interval
  - Bybit codes: "1","3","5","15","30","60","120","240","360","720","D","W","M"
  - example: "60" is for 1-hour candles
- start, end
  - ISO date strings (YYYY-MM-DD format, UTC time)
- ema_range
  - iterable of EMA periods to evaluate
  - examples:
    - range(5, 201, 5) calculates EMA 5, 10, 15, ..., 200
    - range[9, 21, 50, 200] calculates EMA 9, 21, 50, 200
- delta
  - distance between EMA and candle (units depend on delta_mode)
    - delta percent = 0.5 → "within 0.5% of EMA"
    - delta absolute: delta = 40 → "within $40 of EMA"
  - examples:
    - a 5 USDT price distance in % is calculated as:  5 / price (e.g. 67,000) × 100 = 0.00746%
    - a 0.0075% delta in USDT is calculated as: price (e.g. 67,000) × 0.0075 / 100 = 5.025 USDT
- delta_mode
  - "percent":
    - delta is a % of the EMA value at each candle (scales across price regimes)
    - the allowed distance between EMA and candle changes with price
  - "absolute":
    - delta is a fixed price distance in quote currency (distance in price units, e.g. USDT)
    - the allowed distance is always exactly the chosen 'number' regardless of price
- category
  - "linear" for USDT-margined perps
  - "inverse" for coin-margined perps

In [2]:
symbol     = "BTCUSDT"
interval   = "15"
start      = "2026-01-01"
end        = "2026-04-01"
ema_range  = range(1, 200, 1)
delta      = 20
delta_mode = "absolute"
category   = "linear"

save_config(symbol, interval, start, end, ema_range, delta, delta_mode, category)

[ema_core] Saved config: data/config.json


<a id="fetch-ohlc"></a>
## Fetch OHLC

load_or_fetch_klines:
- returns a Dataframe of raw OHLCV candles (klines) from Bybit
- returns the cached parquet if present, otherwise fetches from Bybit and caches the result
- to refresh: pass force_refetch=True

In [3]:
df = load_or_fetch_klines(symbol, interval, start, end, category)
print(f"  -> {len(df)} candles loaded, interval={interval}, {df['timestamp'].iloc[0]} → {df['timestamp'].iloc[-1]}")
df.head()


[ema_core] Fetching BTCUSDT 15 2026-01-01 → 2026-04-01 from ByBit ...
[ema_core] Saved klines cache: klines_btcusdt_15_2026-01-01_2026-04-01_linear.parquet (8641 candles)
  -> 8641 candles loaded, interval=15, 2026-01-01 00:00:00+00:00 → 2026-04-01 00:00:00+00:00


,timestamp,open,high,low,close,volume,turnover
0,2026-01-01 00:00:00+00:00,87594.9,87770.8,87574.0,87684.8,323.475,2.836000e+07
1,2026-01-01 00:15:00+00:00,87684.8,87738.9,87684.7,87723.9,158.001,1.385838e+07
2,2026-01-01 00:30:00+00:00,87723.9,87726.3,87676.2,87698.1,144.601,1.268179e+07
3,2026-01-01 00:45:00+00:00,87698.1,87814.0,87698.1,87758.5,125.589,1.102255e+07
4,2026-01-01 01:00:00+00:00,87758.5,87977.9,87758.5,87950.2,744.620,6.545452e+07


<a id="cache-and-re-fetching"></a>
## Cache and re-fetching

__What gets cached:__ \
Raw OHLC from Bybit: one parquet file under data/klines_{symbol}_{interval}_{start}_{end}_{category}.parquet.

__Default behavior__ (load_or_fetch_klines):
- If a klines file exists for your config → load from cache
- Otherwise → fetch from Bybit and save
- First run (new market): seconds to minutes
- Subsequent runs: instant

__What happens when you change parameters:__
| change | klines cache | what to run |
|---|---|---|
| symbol, interval, start, end, category | new entry | full pipeline (fetch + aggregate) |
| ema_range, delta, delta_mode, skip_warmup | unchanged | aggregate only (no fetch) |

__Cache management:__
- Old klines files are not deleted automatically
- Clean them up with: rm data/*.parquet

__Force a refresh:__
- Use this if data has changed (Bybit corrected historical data) or to test the cache logic itself.
- This re-fetches from Bybit and overwrites the existing cache file: \
  df = load_or_fetch_klines(symbol, interval, start, end, category, force_refetch=True)

<a id="aggregate-ema-metrics"></a>
## Aggregate EMA metrics

analyze_ema_touches algorithm:
1. For every EMA period in ema_range:
   - Computes the EMA on close prices.
   - Assigns every evaluated candle to a category.
   - Categories are mutually exclusive.
   - Counts support/resistance tests and how many hold, across all evaluated candles.
2. Returns a DataFrame with one row per EMA period.

Partition table: every evaluated candle is placed in exactly one category.

| category     | condition                                                            | behavior | meaning |
|--------------|----------------------------------------------------------------------|---------|----------|
| cross      | Low ≤ EMA ≤ High | candle crosses EMA | level broken             |
| low_touch  | Low > EMA AND Low − EMA ≤ delta | entire candle above EMA, Low within delta of EMA, Low approached EMA from above | support held, no break |
| high_touch | High < EMA AND EMA − High ≤ delta | entire candle below EMA, High within delta of EMA, High approached EMA from below | resistance held, no break |
| above      | Low > EMA AND Low − EMA > delta | entire candle above EMA, Low further than delta | clean uptrend, no test of EMA |
| below      | High < EMA AND EMA − High > delta | entire candle below EMA, High further than delta | clean downtrend, no test of EMA |

Invariants:
- cross + low_touch + high_touch + above + below = evaluated_candles
- any_touch = low_touch + high_touch (mutually exclusive — a candle either tests support OR resistance, never both)


In [4]:
result = analyze_ema_touches(df, ema_range, delta, delta_mode)
print(f"  -> {len(result)} EMA periods analyzed.")
result.head(10).style.hide(axis="index")


  -> 199 EMA periods analyzed.


ema,low_touch,high_touch,any_touch,above,below,cross,cross_above,cross_below,support_test,support_held,resistance_test,resistance_held,evaluated_candles
1,0,0,0,0,0,8640,4312,4325,4312,0,4325,0,8640
2,30,33,63,3,3,8570,4289,4281,4319,756,4314,718,8639
3,159,130,289,110,103,8136,4086,4050,4245,1354,4180,1281,8638
4,289,239,528,362,351,7396,3722,3674,4011,1658,3913,1588,8637
5,369,346,715,673,633,6615,3327,3288,3696,1724,3634,1728,8636
6,375,405,780,997,943,5915,2983,2932,3358,1688,3337,1724,8635
7,433,426,859,1231,1241,5303,2660,2643,3093,1621,3069,1643,8634
8,415,399,814,1470,1502,4847,2411,2436,2826,1513,2835,1519,8633
9,399,393,792,1648,1684,4508,2248,2260,2647,1439,2653,1467,8632
10,411,383,794,1811,1858,4168,2072,2096,2483,1356,2479,1389,8631


__Alternative: convenience wrapper__

Convenience wrapper:
- chains fetch + aggregate in one call
- does NOT use the parquet cache (always fetches from Bybit)
- for cached fetch, use load_or_fetch_klines + analyze_ema_touches instead (what this notebook does)
- returns 2 dataframes: df, result


In [ ]:
#from ema_core import run
#df, result = run(symbol, interval, start, end, ema_range, delta, delta_mode, category)

<a id="sanity-check"></a>
## Sanity check

A quick look at the partition counts for a handful of EMAs. \
The invariant cross + low_touch + high_touch + above + below = evaluated_candles should hold for every row.

In [5]:
sample = result.iloc[::max(1, len(result) // 10)][['ema', 'low_touch', 'high_touch', 'cross', 'above', 'below', 'evaluated_candles']]
sample.assign(_sum=sample[['low_touch','high_touch','cross','above','below']].sum(axis=1)).head(10)


,ema,low_touch,high_touch,cross,above,below,evaluated_candles,_sum
0,1,0,0,8640,0,0,8640,8640
19,20,234,313,2677,2615,2782,8621,8621
38,39,174,184,1813,3043,3388,8602,8602
57,58,133,162,1355,3247,3686,8583,8583
76,77,113,125,1136,3334,3856,8564,8564
95,96,98,115,1013,3347,3972,8545,8545
114,115,119,93,913,3331,4070,8526,8526
133,134,77,77,863,3312,4178,8507,8507
152,153,79,84,821,3243,4261,8488,8488
171,172,62,69,773,3214,4351,8469,8469


<a id="data-export"></a>
## Data export

Saving dataframes to CSV for further analysis in Excel or Tableau.

In [6]:
filename_df = f"df_{symbol}_{interval}_{start}_{end}.csv"
df.to_csv(filename_df, index=False)
print(f"\nResults saved to {filename_df}")


Results saved to df_BTCUSDT_15_2026-01-01_2026-04-01.csv


In [7]:
filename_ema = f"ema_analysis_{symbol}_{interval}_{start}_{end}_{ema_range}_{delta}.csv"
result.to_csv(filename_ema, index=False)
print(f"\nResults saved to {filename_ema}")


Results saved to ema_analysis_BTCUSDT_15_2026-01-01_2026-04-01_range(1, 200)_20.csv


<a id="next-steps"></a>
## Next steps

1. Klines 'df' and the config file are saved to data/.
2. Dataframe 'result' is not saved — it’s recomputed each session.
3. Notebooks 2 and 3 automatically load the cached OHLC and the config, and recompute 'result'.
4. Open 2_ema_analysis.ipynb next to interpret which EMAs act as support / resistance / universal S/R.
5. Or skip ahead to 3_ema_backtesting.ipynb to backtest a strategy on this dataset.

__Practical workflow__

| if you want | run |
|---|---|
| to explore the data without trading anything | 1 → 2 |
| to understand the EMA landscape before committing to a strategy | 1 → 2 → 3 |
| to check whether a strategy is profitable on this market right now | 1 → 3 (skip 2) |
